In [11]:
import numpy as np
from sklearn.datasets import load_iris

def sigmoid(z):
    return 1.0/ (1.0 + np.exp(-np.clip(z, -250, 250)))

def bce_from_logits(z, y):
    """-[y log p + (1-y) log(1-p)] rewritten as log(1+e^z) - y*z as an equivalent algebraic expression for numerical stability

    Never log(sigmoid(z)): that is log(0) waiting to happen
    """

    return np.mean(np.logaddexp(0.0, z) - y * z)

class LogisticRegressionGD:
    """y in {0, 1}. loss='bce' is the real thing 'mse' is here to fail."""

    def __init__(self, lr=0.1, epochs=500,loss="bce", seed=1):
        self.lr, self.epochs, self.loss, self.seed = lr, epochs, loss, seed

    def net_input(self, X):
        return X @ self.w_ + self.b_

    def predict_proba(self, X):
        return sigmoid(self.net_input(X))

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    def fit(self, X, y, init_w=None):
        rng = np.random.default_rng(self.seed)
        if init_w is None:
           self.w_ = rng.normal(scale=0.01, size=X.shape[1])
        else:
            self.w_ = np.array(init_w, dtype=float)
        self.b_, self.losses_, self.wnorm_ = 0.0, [], []
        n = X.shape[0]
        for _ in range(self.epochs):
            z = self.net_input(X)
            p = sigmoid(z)
            self.losses_.append(bce_from_logits(z, y)) # always report BCE
            self.wnorm_.append(np.linalg.norm(z))
            if self.loss == 'bce':
                dz = (p - y) # the whole gradient term, no chain term
            else:
                dz = 2.0 * (p- y) * p * (1.0 - p) # MSE drags p(1-p) along
            self.w_ -= self.lr * (X.T @ dz) / n
            self.b_ -= self.lr * dz.mean()
        return self


def iris_pair(a, b, cols=(2,3)):
    """Two iris classes, standardized features, labels 0/1."""
    d = load_iris()
    mask = np.isin(d.target, [a, b])
    X = d.data[mask][:, list(cols)]
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    return X, (d.target[mask] == b).astype(float)

if __name__ == "__main__":
    # versicolor vs. virginica: overlapping, so the optimum is finite

    X, y = iris_pair(1, 2)
    clf = LogisticRegressionGD(lr=0.5, epochs=2000).fit(X, y)
    print("loss first -> last:", round(clf.losses_[0], 4), "->",
          round(clf.losses_[-1], 4))
    print("weights, bias    :", np.round(clf.w_, 4), round(clf.b_, 4))
    print("train accuracy   :", np.mean(clf.predict(X) == y))

    # 1. the same model, fitted with squared error instead of log loss
    #     started from a confidently wrong place

    bad = [-4.0, -4.0]
    bce = LogisticRegressionGD(lr=0.5, epochs=200, loss="bce").fit(X, y, init_w=bad)
    mse = LogisticRegressionGD(lr=0.5, epochs=200, loss="mse").fit(X, y, init_w=bad)
    print("from a bad start, BCE loss after 200 epochs:",
          round(bce.losses_[-1], 4), "acc", np.mean(bce.predict(X) == y))
    print("from a bad start, MSE loss after 200 epochs:",
          round(mse.losses_[-1], 4), "acc", np.mean(mse.predict(X) == y))


    # 2. perfectly separable data has no finite maximum likelihood
    Xs, ys = iris_pair(0, 1);
    for n_epochs in (100, 1000, 10000):
        run = LogisticRegressionGD(lr=0.5, epochs=n_epochs).fit(Xs, ys)
        print(f"separable, {n_epochs:>5} epochs: loss {run.losses_[-1]:.2e}"
              f" ||w|| {np.linalg.norm(run.w_):.2f}")

    # 3. diff against the reference (C huge = effectively no penalty)

    from sklearn.linear_model import LogisticRegression
    sk = LogisticRegression(C=1e6, max_iter=10000).fit(X, y)
    print("mine      :", np.round(clf.w_, 3), round(clf.b_, 3))
    print("sklearn:", np.round(sk.coef_[0], 3), round(float(sk.intercept_[0]), 3))
    assert np.allclose(clf.predict_proba(X), sk.predict_proba(X)[:, 1], atol=0.02)
    print("probabilities match sklearn within 0.02")




loss first -> last: 0.6884 -> 0.1028
weights, bias    : [4.6131 4.3668] 0.4613
train accuracy   : 0.94
from a bad start, BCE loss after 200 epochs: 0.1141 acc 0.94
from a bad start, MSE loss after 200 epochs: 4.1394 acc 0.08
separable,   100 epochs: loss 1.89e-02 ||w|| 3.41
separable,  1000 epochs: loss 2.61e-03 ||w|| 5.60
separable, 10000 epochs: loss 3.27e-04 ||w|| 8.16
mine      : [4.613 4.367] 0.461
sklearn: [4.721 4.423] 0.47
probabilities match sklearn within 0.02


In [5]:
import numpy as np
np.isin(d.target, [0,1])

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,